In [1]:
#Imports and directory
import os
import pickle
import random
import math
import numpy as np
import pandas as pd
import matplotlib
import torch
import torch.nn as nn
from matplotlib import pyplot as plt
#os.chdir("/home/ec2-user/CS-230-Deep-Learning-Project")
os.chdir(r"C:\VScode\Projet Stanford CS230\CS-230-Deep-Learning-Project")
#os.chdir(r"C:\Users\gotta\OneDrive\Documents\Bureau\X\4A\US\Stanford\Classes\CS 230\Project\CS-230-Deep-Learning-Project")

In [2]:
# Basic functions including device and seeding
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
print("USING DEVICE:", device)

def save_object_to_file(obj, filepath):
    with open(filepath, "wb") as f:
        pickle.dump(obj, f)

def read_object_from_file(filepath):
    with open(filepath, "rb") as f:
        return pickle.load(f)

def seed_everything(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


USING DEVICE: cpu


In [3]:
# --- Data preparation ---
DATA_CSV_FR = os.path.join("data", "FR_price", "FR_lmp_processed.csv")
DATA_CSV_LOAD = os.path.join("data", "FR_DAM_Load", "FR_load_processed.csv")
DATA_CSV_REN = os.path.join("data", "FR_REN", "FR_ren_processed.csv")
DATA_CSV_LOAD_RT=os.path.join("data", "FR_DAM_Load", "FR_rt_load_processed.csv")
DATA_CSV_REN_RT=os.path.join("data", "FR_REN", "FR_rt_ren_processed.csv")


In [4]:
df_FR = pd.read_csv(DATA_CSV_FR)
load_FR = pd.read_csv(DATA_CSV_LOAD)
ren_FR = pd.read_csv(DATA_CSV_REN)
all_data_DA = pd.merge(df_FR, load_FR, on=['Time', 'Timezone', 'Year', 'Month', 'Day', 'Hour'], how='inner')
all_data_DA = pd.merge(all_data_DA, ren_FR, on=['Time', 'Timezone', 'Year', 'Month', 'Day', 'Hour'], how='inner')
all_data_DA.rename(columns={'MW':'Load_DA','Wind Onshore':'Wind_DA','Solar':'Solar_DA'}, inplace=True)
load_FR_rt = pd.read_csv(DATA_CSV_LOAD_RT)
ren_FR_rt = pd.read_csv(DATA_CSV_REN_RT)
all_data_RT = pd.merge(df_FR, load_FR_rt, on=['Time', 'Timezone', 'Year', 'Month', 'Day', 'Hour'], how='inner')
all_data_RT = pd.merge(all_data_RT, ren_FR_rt, on=['Time', 'Timezone', 'Year', 'Month', 'Day', 'Hour'], how='inner')
all_data_RT.rename(columns={'MW':'Load_RT','Wind Onshore':'Wind_RT','Solar':'Solar_RT'}, inplace=True)

all_data=pd.merge(all_data_DA,all_data_RT,on=['Time', 'Timezone', 'Year', 'Month', 'Day', 'Hour','EUR/MWh'], how='inner')
all_data

,Time,Year,Month,Day,Hour,Timezone,EUR/MWh,Load_DA,Wind_DA,Solar_DA,Load_RT,Wind_RT,Solar_RT
0,2019-10-01 00:00:00+02:00,2019,10,1,0,+02:00,33.09,44450.0,4975.31,0.00,43062.00,6076.00,0.00
1,2019-10-01 01:00:00+02:00,2019,10,1,1,+02:00,27.72,41300.0,5338.58,0.00,40483.00,6137.00,0.00
2,2019-10-01 02:00:00+02:00,2019,10,1,2,+02:00,23.12,40050.0,5702.33,0.00,39207.00,6342.00,0.00
3,2019-10-01 03:00:00+02:00,2019,10,1,3,+02:00,16.46,37750.0,5667.94,0.00,37004.00,6355.00,0.00
4,2019-10-01 04:00:00+02:00,2019,10,1,4,+02:00,15.66,36600.0,5633.30,0.00,36399.00,6217.00,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...
52142,2025-09-30 19:00:00+02:00,2025,9,30,19,+02:00,117.00,49000.0,2435.69,2158.72,49421.41,1939.43,901.85
52143,2025-09-30 20:00:00+02:00,2025,9,30,20,+02:00,106.18,50400.0,2929.19,315.89,50315.77,2445.66,244.45
52144,2025-09-30 21:00:00+02:00,2025,9,30,21,+02:00,83.23,46700.0,3480.47,0.00,47039.08,3116.56,0.00
52145,2025-09-30 22:00:00+02:00,2025,9,30,22,+02:00,73.00,44100.0,3879.11,0.00,45544.52,3422.12,0.00


In [5]:
#Convert hour, day, month into cyclic features
def add_cyclical_features(df):
    df['Date']=pd.to_datetime(df[['Year', 'Month', 'Day', 'Hour']])
    df["DayOfWeek"] = df['Date'].dt.dayofweek
    # Hour of day (0-23)
    df["sin_hour"] = np.sin(2 * np.pi * df["Hour"] / 24)
    df["cos_hour"] = np.cos(2 * np.pi * df["Hour"] / 24)

    # Day of week (0-6 or 1-7 depending on your data)
    df["sin_day"] = np.sin(2 * np.pi * df["DayOfWeek"] / 7)
    df["cos_day"] = np.cos(2 * np.pi * df["DayOfWeek"] / 7)

    # Month (1-12)
    df["sin_month"] = np.sin(2 * np.pi * df["Month"] / 12)
    df["cos_month"] = np.cos(2 * np.pi * df["Month"] / 12)
    return df
all_data = add_cyclical_features(all_data)

all_data


,Time,Year,Month,Day,Hour,Timezone,EUR/MWh,Load_DA,Wind_DA,Solar_DA,...,Wind_RT,Solar_RT,Date,DayOfWeek,sin_hour,cos_hour,sin_day,cos_day,sin_month,cos_month
0,2019-10-01 00:00:00+02:00,2019,10,1,0,+02:00,33.09,44450.0,4975.31,0.00,...,6076.00,0.00,2019-10-01 00:00:00,1,0.000000,1.000000,0.781831,0.62349,-0.866025,5.000000e-01
1,2019-10-01 01:00:00+02:00,2019,10,1,1,+02:00,27.72,41300.0,5338.58,0.00,...,6137.00,0.00,2019-10-01 01:00:00,1,0.258819,0.965926,0.781831,0.62349,-0.866025,5.000000e-01
2,2019-10-01 02:00:00+02:00,2019,10,1,2,+02:00,23.12,40050.0,5702.33,0.00,...,6342.00,0.00,2019-10-01 02:00:00,1,0.500000,0.866025,0.781831,0.62349,-0.866025,5.000000e-01
3,2019-10-01 03:00:00+02:00,2019,10,1,3,+02:00,16.46,37750.0,5667.94,0.00,...,6355.00,0.00,2019-10-01 03:00:00,1,0.707107,0.707107,0.781831,0.62349,-0.866025,5.000000e-01
4,2019-10-01 04:00:00+02:00,2019,10,1,4,+02:00,15.66,36600.0,5633.30,0.00,...,6217.00,0.00,2019-10-01 04:00:00,1,0.866025,0.500000,0.781831,0.62349,-0.866025,5.000000e-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
52142,2025-09-30 19:00:00+02:00,2025,9,30,19,+02:00,117.00,49000.0,2435.69,2158.72,...,1939.43,901.85,2025-09-30 19:00:00,1,-0.965926,0.258819,0.781831,0.62349,-1.000000,-1.836970e-16
52143,2025-09-30 20:00:00+02:00,2025,9,30,20,+02:00,106.18,50400.0,2929.19,315.89,...,2445.66,244.45,2025-09-30 20:00:00,1,-0.866025,0.500000,0.781831,0.62349,-1.000000,-1.836970e-16
52144,2025-09-30 21:00:00+02:00,2025,9,30,21,+02:00,83.23,46700.0,3480.47,0.00,...,3116.56,0.00,2025-09-30 21:00:00,1,-0.707107,0.707107,0.781831,0.62349,-1.000000,-1.836970e-16
52145,2025-09-30 22:00:00+02:00,2025,9,30,22,+02:00,73.00,44100.0,3879.11,0.00,...,3422.12,0.00,2025-09-30 22:00:00,1,-0.500000,0.866025,0.781831,0.62349,-1.000000,-1.836970e-16


In [73]:
#normalization
# Columns to exclude
exclude_cols = ["Time", "Timezone", "Year", "Month", "Day", "Hour", 'Date', 'DayOfWeek', 'cos_hour', 'sin_hour', 'cos_day', 'sin_day', 'cos_month', 'sin_month']

# Columns we want to normalize
norm_cols = [c for c in all_data.columns if c not in exclude_cols]

# Compute stats only on selected columns
means = all_data[norm_cols].mean()
medians = all_data[norm_cols].median()
stds  = all_data[norm_cols].std()

# Apply Gaussian normalization only to norm_cols
all_data_norm = all_data.copy()
all_data_norm[norm_cols] = (all_data[norm_cols] - means) / stds

all_data = all_data_norm

print(means)
print(stds)

EUR/MWh       104.796131
Load_DA     50508.327325
Wind_DA      4708.093065
Solar_DA     2291.111826
Load_RT     50465.382685
Wind_RT      4540.707101
Solar_RT     2209.322986
dtype: float64
EUR/MWh       110.919744
Load_DA     11085.904859
Wind_DA      3484.189322
Solar_DA     3486.011115
Load_RT     10901.326210
Wind_RT      3295.262375
Solar_RT     3266.016272
dtype: float64


In [74]:
save_path = os.path.join("data/Datasets_2", "normalization_stats.pkl")

with open(save_path, "wb") as f:
    pickle.dump({
        "means": means,
        "stds": stds,
        "medians": medians,
        "norm_cols": norm_cols
    }, f)

print(f"Saved normalization stats to: {save_path}")

Saved normalization stats to: data/Datasets_2/normalization_stats.pkl


In [6]:
def create_sliding_window(data_df, string, window_days=7, forecast_horizon=1):
    """
    Outputs five DataFrames:
    - X_df: past window (arrays) + date
    - y_df: target next-day (arrays) + date
    - y_day_df: previous-day same-hour baseline + date
    - y_week_df: previous-week same-hour baseline + date
    - pct_df: percent change vs last window value + date
    """


    series = data_df[string].values
    dates = pd.to_datetime(data_df["Date"])

    hours_per_day = 24
    window_size = window_days * hours_per_day
    eps = 1e-9

    X, y = [], []
    y_day_before, y_week_before = [], []
    y_pct = []
    y_dates = []

    n = len(series)

    for i in range(0, n - window_size - forecast_horizon + 1):

        # Input window
        X_window = series[i : i + window_size]
        # Forecast target
        y_target = series[i + window_size : i + window_size + forecast_horizon]

        # Target date
        date_target = dates[i + window_size]

        # Previous-day baseline
        start_prev_day = i + window_size - hours_per_day
        if start_prev_day >= 0:
            prev_day = series[start_prev_day : start_prev_day + forecast_horizon]
        else:
            prev_day = np.full(forecast_horizon, np.nan)

        # Previous-week baseline
        start_prev_week = i + window_size - 7 * hours_per_day
        if start_prev_week >= 0:
            prev_week = series[start_prev_week : start_prev_week + forecast_horizon]
        else:
            prev_week = np.full(forecast_horizon, np.nan)

        # Percent change wrt last hour in window
        pct = (y_target - X_window[-1]) / (X_window[-1] + eps)

        # Append everything
        X.append(X_window.astype(np.float32))
        y.append(y_target.astype(np.float32))
        y_day_before.append(prev_day.astype(np.float32))
        y_week_before.append(prev_week.astype(np.float32))
        y_pct.append(pct.astype(np.float32))
        y_dates.append(date_target)

    # ---- Build DataFrames ----
    X_df = pd.DataFrame({"X": X, "Date": y_dates})
    y_df = pd.DataFrame({"y": y, "Date": y_dates})
    y_day_df = pd.DataFrame({"y_day": y_day_before, "Date": y_dates})
    y_week_df = pd.DataFrame({"y_week": y_week_before, "Date": y_dates})
    pct_df = pd.DataFrame({"pct": y_pct, "Date": y_dates})

    return X_df, y_df, y_day_df, y_week_df, pct_df


In [7]:
#Create sliding windows
X_df, y_df, y_day_df, y_week_df, y_per_df   = create_sliding_window(all_data,'EUR/MWh', window_days=7, forecast_horizon=1)
X_load_DA_df, _, _, _, _ = create_sliding_window(all_data, string='Load_DA', window_days=7)
X_wind_DA_df, _, _, _,_ = create_sliding_window(all_data, string='Wind_DA', window_days=7)
X_solar_DA_df, _, _, _,_ = create_sliding_window(all_data, string='Solar_DA', window_days=7)
X_cos_hour_df, _, _, _, _ = create_sliding_window(all_data, string='cos_hour', window_days=7)
X_sin_hour_df, _, _, _,_ = create_sliding_window(all_data, string='sin_hour', window_days=7)
X_cos_day_df, _, _, _,_ = create_sliding_window(all_data, string='cos_day', window_days=7)
X_sin_day_df, _, _, _,_ = create_sliding_window(all_data, string='sin_day', window_days=7)
X_cos_month_df, _, _, _,_ = create_sliding_window(all_data, string='cos_month', window_days=7)
X_sin_month_df, _, _, _,_ = create_sliding_window(all_data, string='sin_month', window_days=7)

X_df = X_df.rename(columns={'X': 'EUR/MWh'})
X_load_DA_df = X_load_DA_df.rename(columns={'X': 'Load_DA'})
X_wind_DA_df= X_wind_DA_df.rename(columns={'X': 'Wind_DA'})
X_solar_DA_df = X_solar_DA_df.rename(columns={'X': 'Solar_DA'})
X_cos_hour_df = X_cos_hour_df.rename(columns={'X': 'cos_hour'})
X_sin_hour_df= X_sin_hour_df.rename(columns={'X': 'sin_hour'})
X_cos_day_df = X_cos_day_df.rename(columns={'X': 'cos_day'})
X_sin_day_df = X_sin_day_df.rename(columns={'X': 'sin_day'})
X_cos_month_df= X_cos_month_df.rename(columns={'X': 'cos_month'})
X_sin_month_df = X_sin_month_df.rename(columns={'X': 'sin_month'})


In [8]:
X_solar_DA_df

,Solar_DA,Date
0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.33,...",2019-10-08 00:00:00
1,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.33, 524....",2019-10-08 01:00:00
2,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.33, 524.54, 1...",2019-10-08 02:00:00
3,"[0.0, 0.0, 0.0, 0.0, 0.0, 1.33, 524.54, 1775.8...",2019-10-08 03:00:00
4,"[0.0, 0.0, 0.0, 0.0, 1.33, 524.54, 1775.88, 29...",2019-10-08 04:00:00
...,...,...
51974,"[4907.61, 2134.87, 383.22, 0.3, 0.0, 0.0, 0.0,...",2025-09-30 19:00:00
51975,"[2134.87, 383.22, 0.3, 0.0, 0.0, 0.0, 0.0, 0.0...",2025-09-30 20:00:00
51976,"[383.22, 0.3, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0....",2025-09-30 21:00:00
51977,"[0.3, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",2025-09-30 22:00:00


In [9]:
# Merge all relevant DataFrames on 'Date'
all_data_df = (
    X_df
    .merge(y_df, on='Date')
    .merge(X_load_DA_df, on='Date')
    .merge(X_wind_DA_df, on='Date')
    .merge(X_solar_DA_df, on='Date')
    .merge(X_cos_hour_df, on='Date')
    .merge(X_sin_hour_df, on='Date')
    .merge(X_cos_day_df, on='Date')
    .merge(X_sin_day_df, on='Date')
    .merge(X_cos_month_df, on='Date')
    .merge(X_sin_month_df, on='Date')
    .merge(y_day_df, on='Date')
    .merge(y_week_df, on='Date')
    .merge(y_per_df, on='Date')
)

# Reorder and select only the desired columns
all_data_df = all_data_df[['Date', 'EUR/MWh', 'Load_DA', 'Wind_DA', 'Solar_DA', 'cos_hour','sin_hour',	'cos_day','sin_day', 'cos_month',	'sin_month','y', 'y_day', 'y_week', 'pct']]
all_data_df

,Date,EUR/MWh,Load_DA,Wind_DA,Solar_DA,cos_hour,sin_hour,cos_day,sin_day,cos_month,sin_month,y,y_day,y_week,pct
0,2019-10-08 00:00:00,"[33.09, 27.72, 23.12, 16.46, 15.66, 21.39, 38....","[44450.0, 41300.0, 40050.0, 37750.0, 36600.0, ...","[4975.31, 5338.58, 5702.33, 5667.94, 5633.3, 5...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.33,...","[1.0, 0.9659258, 0.8660254, 0.70710677, 0.5, 0...","[0.0, 0.25881904, 0.5, 0.70710677, 0.8660254, ...","[0.6234898, 0.6234898, 0.6234898, 0.6234898, 0...","[0.7818315, 0.7818315, 0.7818315, 0.7818315, 0...","[0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, ...","[-0.8660254, -0.8660254, -0.8660254, -0.866025...",[30.64],[28.99],[33.09],[-0.06499848]
1,2019-10-08 01:00:00,"[27.72, 23.12, 16.46, 15.66, 21.39, 38.51, 50....","[41300.0, 40050.0, 37750.0, 36600.0, 37800.0, ...","[5338.58, 5702.33, 5667.94, 5633.3, 5599.0, 56...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.33, 524....","[0.9659258, 0.8660254, 0.70710677, 0.5, 0.2588...","[0.25881904, 0.5, 0.70710677, 0.8660254, 0.965...","[0.6234898, 0.6234898, 0.6234898, 0.6234898, 0...","[0.7818315, 0.7818315, 0.7818315, 0.7818315, 0...","[0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, ...","[-0.8660254, -0.8660254, -0.8660254, -0.866025...",[27.93],[27.07],[27.72],[-0.088446476]
2,2019-10-08 02:00:00,"[23.12, 16.46, 15.66, 21.39, 38.51, 50.03, 50....","[40050.0, 37750.0, 36600.0, 37800.0, 41950.0, ...","[5702.33, 5667.94, 5633.3, 5599.0, 5603.01, 56...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.33, 524.54, 1...","[0.8660254, 0.70710677, 0.5, 0.25881904, 6.123...","[0.5, 0.70710677, 0.8660254, 0.9659258, 1.0, 0...","[0.6234898, 0.6234898, 0.6234898, 0.6234898, 0...","[0.7818315, 0.7818315, 0.7818315, 0.7818315, 0...","[0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, ...","[-0.8660254, -0.8660254, -0.8660254, -0.866025...",[25.0],[27.0],[23.12],[-0.10490512]
3,2019-10-08 03:00:00,"[16.46, 15.66, 21.39, 38.51, 50.03, 50.97, 49....","[37750.0, 36600.0, 37800.0, 41950.0, 48100.0, ...","[5667.94, 5633.3, 5599.0, 5603.01, 5607.08, 56...","[0.0, 0.0, 0.0, 0.0, 0.0, 1.33, 524.54, 1775.8...","[0.70710677, 0.5, 0.25881904, 6.123234e-17, -0...","[0.70710677, 0.8660254, 0.9659258, 1.0, 0.9659...","[0.6234898, 0.6234898, 0.6234898, 0.6234898, 0...","[0.7818315, 0.7818315, 0.7818315, 0.7818315, 0...","[0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, ...","[-0.8660254, -0.8660254, -0.8660254, -0.866025...",[22.98],[25.77],[16.46],[-0.0808]
4,2019-10-08 04:00:00,"[15.66, 21.39, 38.51, 50.03, 50.97, 49.63, 46....","[36600.0, 37800.0, 41950.0, 48100.0, 50250.0, ...","[5633.3, 5599.0, 5603.01, 5607.08, 5611.07, 57...","[0.0, 0.0, 0.0, 0.0, 1.33, 524.54, 1775.88, 29...","[0.5, 0.25881904, 6.123234e-17, -0.25881904, -...","[0.8660254, 0.9659258, 1.0, 0.9659258, 0.86602...","[0.6234898, 0.6234898, 0.6234898, 0.6234898, 0...","[0.7818315, 0.7818315, 0.7818315, 0.7818315, 0...","[0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, ...","[-0.8660254, -0.8660254, -0.8660254, -0.866025...",[20.71],[24.91],[15.66],[-0.09878155]
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
101120,2025-09-30 19:00:00,"[26.47, 58.0, 40.63, 18.63, 21.96, 21.02, 0.0,...","[45600.0, 48400.0, 49600.0, 46800.0, 44100.0, ...","[7572.2, 6997.82, 6779.35, 6431.93, 6195.38, 6...","[4907.61, 2134.87, 383.22, 0.3, 0.0, 0.0, 0.0,...","[-1.8369701e-16, 0.25881904, 0.5, 0.70710677, ...","[-1.0, -0.9659258, -0.8660254, -0.70710677, -0...","[0.6234898, 0.6234898, 0.6234898, 0.6234898, 0...","[0.7818315, 0.7818315, 0.7818315, 0.7818315, 0...","[-1.8369701e-16, -1.8369701e-16, -1.8369701e-1...","[-1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1....",[117.0],[85.0],[26.47],[0.4098084]
101121,2025-09-30 20:00:00,"[58.0, 40.63, 18.63, 21.96, 21.02, 0.0, 0.0, 0...","[48400.0, 49600.0, 46800.0, 44100.0, 46000.0, ...","[6997.82, 6779.35, 6431.93, 6195.38, 6336.68, ...","[2134.87, 383.22, 0.3, 0.0, 0.0, 0.0, 0.0, 0.0...","[0.25881904, 0.5, 0.70710677, 0.8660254, 0.965...","[-0.9659258, -0.8660254, -0.70710677, -0.5, -0...","[0.6234

In [10]:
SEED = 42
seed_everything(SEED)

In [11]:
# --- Train, Dev, Test Split based on randomly splitting by month---
def month_based_split(df, train_ratio=0.6, dev_ratio=0.2, seed=42):

    df = df.copy()
    df["Date"] = pd.to_datetime(df["Date"])

    # Create a unique month identifier
    df["YearMonth"] = df["Date"].dt.to_period("M")

    # List of unique months
    all_months = df["YearMonth"].unique()
    n_months = len(all_months)

    # Shuffle months
    rng = np.random.default_rng(seed)
    shuffled_months = rng.permutation(all_months)

    # Compute split sizes (in months)
    n_train = int(train_ratio * n_months)
    n_dev   = int(dev_ratio   * n_months)


    # Assign months
    train_months = shuffled_months[:n_train]
    dev_months   = shuffled_months[n_train : n_train + n_dev]
    test_months  = shuffled_months[n_train + n_dev :]

    # Build the splits
    train_df = df[df["YearMonth"].isin(train_months)].drop(columns="YearMonth")
    dev_df   = df[df["YearMonth"].isin(dev_months)].drop(columns="YearMonth")
    test_df  = df[df["YearMonth"].isin(test_months)].drop(columns="YearMonth")

    return train_df, dev_df, test_df, train_months, dev_months, test_months


In [12]:
train_df, dev_df, test_df, train_months, dev_months, test_months = month_based_split(all_data_df)

print("TRAIN months:", train_months)
print("DEV months:", dev_months)
print("TEST months:", test_months)

print(len(train_df), len(dev_df), len(test_df))


TRAIN months: [Period('2023-10', 'M') Period('2022-02', 'M') Period('2021-04', 'M')
 Period('2021-11', 'M') Period('2023-01', 'M') Period('2022-11', 'M')
 Period('2024-10', 'M') Period('2023-12', 'M') Period('2022-07', 'M')
 Period('2022-03', 'M') Period('2024-07', 'M') Period('2025-01', 'M')
 Period('2023-06', 'M') Period('2021-03', 'M') Period('2020-02', 'M')
 Period('2021-10', 'M') Period('2021-07', 'M') Period('2020-05', 'M')
 Period('2021-12', 'M') Period('2025-02', 'M') Period('2025-09', 'M')
 Period('2024-06', 'M') Period('2025-07', 'M') Period('2020-03', 'M')
 Period('2019-11', 'M') Period('2022-01', 'M') Period('2024-01', 'M')
 Period('2019-12', 'M') Period('2023-02', 'M') Period('2025-08', 'M')
 Period('2025-06', 'M') Period('2021-06', 'M') Period('2024-08', 'M')
 Period('2022-06', 'M') Period('2023-04', 'M') Period('2021-09', 'M')
 Period('2024-04', 'M') Period('2021-01', 'M') Period('2020-07', 'M')
 Period('2022-05', 'M') Period('2023-05', 'M') Period('2021-02', 'M')
 Perio

In [16]:
train_df.head()

,Date,EUR/MWh,Load_DA,Wind_DA,Solar_DA,cos_hour,sin_hour,cos_day,sin_day,cos_month,sin_month,y,y_day,y_week,pct
528,2019-11-01 00:00:00,"[35.78, 32.31, 31.36, 30.09, 30.39, 33.5, 40.7...","[50150.0, 46700.0, 45750.0, 42800.0, 41250.0, ...","[4351.9, 4468.06, 4582.97, 4658.58, 4733.26, 4...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[1.0, 0.9659258, 0.8660254, 0.70710677, 0.5, 0...","[0.0, 0.25881904, 0.5, 0.70710677, 0.8660254, ...","[-0.22252093, -0.22252093, -0.22252093, -0.222...","[0.9749279, 0.9749279, 0.9749279, 0.9749279, 0...","[0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, ...","[-0.8660254, -0.8660254, -0.8660254, -0.866025...",[35.94],[38.48],[35.78],[-0.1015]
529,2019-11-01 01:00:00,"[32.31, 31.36, 30.09, 30.39, 33.5, 40.7, 52.4,...","[46700.0, 45750.0, 42800.0, 41250.0, 42550.0, ...","[4468.06, 4582.97, 4658.58, 4733.26, 4807.01, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 67.26...","[0.9659258, 0.8660254, 0.70710677, 0.5, 0.2588...","[0.25881904, 0.5, 0.70710677, 0.8660254, 0.965...","[-0.22252093, -0.22252093, -0.22252093, -0.222...","[0.9749279, 0.9749279, 0.9749279, 0.9749279, 0...","[0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, ...","[-0.8660254, -0.8660254, -0.8660254, -0.866025...",[33.49],[35.97],[32.31],[-0.06816917]
530,2019-11-01 02:00:00,"[31.36, 30.09, 30.39, 33.5, 40.7, 52.4, 59.77,...","[45750.0, 42800.0, 41250.0, 42550.0, 46800.0, ...","[4582.97, 4658.58, 4733.26, 4807.01, 4816.39, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 67.26, 644...","[0.8660254, 0.70710677, 0.5, 0.25881904, 6.123...","[0.5, 0.70710677, 0.8660254, 0.9659258, 1.0, 0...","[-0.22252093, -0.22252093, -0.22252093, -0.222...","[0.9749279, 0.9749279, 0.9749279, 0.9749279, 0...","[0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, ...","[-0.8660254, -0.8660254, -0.8660254, -0.866025...",[28.99],[33.97],[31.36],[-0.13436846]
531,2019-11-01 03:00:00,"[30.09, 30.39, 33.5, 40.7, 52.4, 59.77, 57.12,...","[42800.0, 41250.0, 42550.0, 46800.0, 52150.0, ...","[4658.58, 4733.26, 4807.01, 4816.39, 4825.5, 4...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 67.26, 644.26, ...","[0.70710677, 0.5, 0.25881904, 6.123234e-17, -0...","[0.70710677, 0.8660254, 0.9659258, 1.0, 0.9659...","[-0.22252093, -0.22252093, -0.22252093, -0.222...","[0.9749279, 0.9749279, 0.9749279, 0.9749279, 0...","[0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, ...","[-0.8660254, -0.8660254, -0.8660254, -0.866025...",[27.12],[33.16],[30.09],[-0.064505]
532,2019-11-01 04:00:00,"[30.39, 33.5, 40.7, 52.4, 59.77, 57.12, 54.11,...","[41250.0, 42550.0, 46800.0, 52150.0, 56450.0, ...","[4733.26, 4807.01, 4816.39, 4825.5, 4833.72, 4...","[0.0, 0.0, 0.0, 0.0, 0.0, 67.26, 644.26, 1498....","[0.5, 0.25881904, 6.123234e-17, -0.25881904, -...","[0.8660254, 0.9659258, 1.0, 0.9659258, 0.86602...","[-0.22252093, -0.22252093, -0.22252093, -0.222...","[0.9749279, 0.9749279, 0.9749279, 0.9749279, 0...","[0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, ...","[-0.8660254, -0.8660254, -0.8660254, -0.866025...",[26.17],[32.54],[30.39],[-0.035029497]


In [30]:
# 1. List of training months (Train Set)
train_months = [
    pd.Period('2023-10', 'M'), pd.Period('2022-02', 'M'), pd.Period('2021-04', 'M'), 
    pd.Period('2021-11', 'M'), pd.Period('2023-01', 'M'), pd.Period('2022-11', 'M'), 
    pd.Period('2024-10', 'M'), pd.Period('2023-12', 'M'), pd.Period('2022-07', 'M'), 
    pd.Period('2022-03', 'M'), pd.Period('2024-07', 'M'), pd.Period('2025-01', 'M'), 
    pd.Period('2023-06', 'M'), pd.Period('2021-03', 'M'), pd.Period('2020-02', 'M'), 
    pd.Period('2021-10', 'M'), pd.Period('2021-07', 'M'), pd.Period('2020-05', 'M'), 
    pd.Period('2021-12', 'M'), pd.Period('2025-02', 'M'), pd.Period('2025-09', 'M'), 
    pd.Period('2024-06', 'M'), pd.Period('2025-07', 'M'), pd.Period('2020-03', 'M'), 
    pd.Period('2019-11', 'M'), pd.Period('2022-01', 'M'), pd.Period('2024-01', 'M'), 
    pd.Period('2019-12', 'M'), pd.Period('2023-02', 'M'), pd.Period('2025-08', 'M'), 
    pd.Period('2025-06', 'M'), pd.Period('2021-06', 'M'), pd.Period('2024-08', 'M'), 
    pd.Period('2022-06', 'M'), pd.Period('2023-04', 'M'), pd.Period('2021-09', 'M'), 
    pd.Period('2024-04', 'M'), pd.Period('2021-01', 'M'), pd.Period('2020-07', 'M'), 
    pd.Period('2022-05', 'M'), pd.Period('2023-05', 'M'), pd.Period('2021-02', 'M'), 
    pd.Period('2024-02', 'M')
]

# 2. Columns to exclude from normalization
# We exclude time-related columns and categorical data
cols = ['EUR/MWh', 'Load_DA', 'Wind_DA', 'Solar_DA',]

# Identify columns to normalize dynamically (all numerical columns not in the exclude list)
norm_cols = [c for c in all_data.columns if c in cols]
print(f"Columns selected for normalization: {norm_cols}")

# 3. Create a mask to identify Training rows based on 'Time'
# Ensure 'Time' is in datetime format (handling timezone if necessary)
all_data['Time'] = pd.to_datetime(all_data['Time'], utc=True)

# Convert 'Time' to Monthly Periods to match the 'train_months' list
# .tz_convert(None) removes timezone info to allow comparison with simple Period objects
data_periods = all_data['Time'].dt.tz_convert(None).dt.to_period('M')

# Boolean mask: True if the row belongs to the Training Set
train_mask = data_periods.isin(train_months)

# 4. Compute Statistics (Mean & Std) ONLY on the Train Set
# This prevents data leakage (the model doesn't "see" the future/test data)
train_subset = all_data.loc[train_mask, norm_cols]
means = train_subset.mean()
stds = train_subset.std()
medians_train = train_subset.median()

# 5. Apply Gaussian Normalization to the ENTIRE dataset
# We use the Train Set's mean and std to scale the whole dataset
all_data_norm = all_data.copy()
all_data_norm[norm_cols] = (all_data[norm_cols] - means) / stds


print("\n--- Means (Calculated on Train Set only) ---")
print(means)
print("\n--- Standard Deviations (Calculated on Train Set only) ---")
print(stds)
print("\n--- Medians (Calculated on Train Set only) ---")
print(medians_train)

Columns selected for normalization: ['EUR/MWh', 'Load_DA', 'Wind_DA', 'Solar_DA']

--- Means (Calculated on Train Set only) ---
EUR/MWh       106.581373
Load_DA     51981.092058
Wind_DA      4755.713740
Solar_DA     2319.492853
dtype: float64

--- Standard Deviations (Calculated on Train Set only) ---
EUR/MWh        98.113643
Load_DA     11626.853248
Wind_DA      3531.237439
Solar_DA     3564.745813
dtype: float64

--- Medians (Calculated on Train Set only) ---
EUR/MWh        78.900
Load_DA     50050.000
Wind_DA      3615.970
Solar_DA       25.515
dtype: float64


In [31]:
# 1. List of training months dev months
dev_months= [
    pd.Period('2020-01', 'M'),
    pd.Period('2022-08', 'M'),
    pd.Period('2022-04', 'M'),
    pd.Period('2020-08', 'M'),
    pd.Period('2024-09', 'M'),
    pd.Period('2025-05', 'M'),
    pd.Period('2024-12', 'M'),
    pd.Period('2020-04', 'M'),
    pd.Period('2022-12', 'M'),
    pd.Period('2024-11', 'M'),
    pd.Period('2020-09', 'M'),
    pd.Period('2024-05', 'M'),
    pd.Period('2023-03', 'M'),
    pd.Period('2023-08', 'M')
]

# 2. Columns to exclude from normalization
# We exclude time-related columns and categorical data
dev_cols = ['EUR/MWh', 'Load_DA', 'Wind_DA', 'Solar_DA']

# Identify columns to normalize dynamically (all numerical columns not in the exclude list)
dev_norm_cols = [c for c in all_data.columns if c in dev_cols]
print(f"Columns selected for normalization: {dev_norm_cols}")

# 3. Create a mask to identify Training rows based on 'Time'
# Ensure 'Time' is in datetime format (handling timezone if necessary)
all_data['Time'] = pd.to_datetime(all_data['Time'], utc=True)

# Convert 'Time' to Monthly Periods to match the 'train_months' list
# .tz_convert(None) removes timezone info to allow comparison with simple Period objects
data_periods = all_data['Time'].dt.tz_convert(None).dt.to_period('M')

# Boolean mask: True if the row belongs to the Training Set
dev_mask = data_periods.isin(dev_months)

# 4. Compute Statistics (Mean & Std) ONLY on the Train Set
# This prevents data leakage (the model doesn't "see" the future/test data)
dev_subset = all_data.loc[dev_mask, dev_norm_cols]
medians_dev = dev_subset.median()

print(medians_dev)

Columns selected for normalization: ['EUR/MWh', 'Load_DA', 'Wind_DA', 'Solar_DA']
EUR/MWh        61.36
Load_DA     46300.00
Wind_DA      3394.72
Solar_DA       39.17
dtype: float64


In [32]:
# 1. List of training months dev months
test_months= [
    pd.Period('2021-08', 'M'),
    pd.Period('2021-05', 'M'),
    pd.Period('2022-09', 'M'),
    pd.Period('2019-10', 'M'),
    pd.Period('2023-09', 'M'),
    pd.Period('2023-11', 'M'),
    pd.Period('2023-07', 'M'),
    pd.Period('2020-10', 'M'),
    pd.Period('2024-03', 'M'),
    pd.Period('2020-12', 'M'),
    pd.Period('2020-11', 'M'),
    pd.Period('2025-04', 'M'),
    pd.Period('2022-10', 'M'),
    pd.Period('2025-03', 'M'),
    pd.Period('2020-06', 'M')
]

# 2. Columns to exclude from normalization
# We exclude time-related columns and categorical data
test_cols = ['EUR/MWh', 'Load_DA', 'Wind_DA', 'Solar_DA']

# Identify columns to normalize dynamically (all numerical columns not in the exclude list)
test_norm_cols = [c for c in all_data.columns if c in test_cols]
print(f"Columns selected for normalization: {test_norm_cols}")

# 3. Create a mask to identify Training rows based on 'Time'
# Ensure 'Time' is in datetime format (handling timezone if necessary)
all_data['Time'] = pd.to_datetime(all_data['Time'], utc=True)

# Convert 'Time' to Monthly Periods to match the 'train_months' list
# .tz_convert(None) removes timezone info to allow comparison with simple Period objects
data_periods = all_data['Time'].dt.tz_convert(None).dt.to_period('M')

# Boolean mask: True if the row belongs to the Training Set
test_mask = data_periods.isin(test_months)

# 4. Compute Statistics (Mean & Std) ONLY on the Train Set
# This prevents data leakage (the model doesn't "see" the future/test data)
test_subset = all_data.loc[test_mask, test_norm_cols]
medians_test = test_subset.median()

print(medians_test)

Columns selected for normalization: ['EUR/MWh', 'Load_DA', 'Wind_DA', 'Solar_DA']
EUR/MWh        59.940
Load_DA     47400.000
Wind_DA      3898.245
Solar_DA       17.665
dtype: float64


In [39]:
save_path = os.path.join("data/Datasets_normalized", "normalization_stats.pkl")

with open(save_path, "wb") as f:
    pickle.dump({
        "means": means,
        "stds": stds,
        "medians_train": medians_train,
        "medians_dev": medians_dev,
        "medians_test": medians_test,
        "norm_cols": norm_cols
    }, f)

print(f"Saved normalization stats to: {save_path}")

Saved normalization stats to: data/Datasets_normalized\normalization_stats.pkl


In [37]:
#normalisation of train_df, dev_df, test_df

train_df_norm = train_df.copy()
train_df_norm[norm_cols] = (train_df[norm_cols] - means) / stds
y_cols=['y', 'y_day', 'y_week']
train_df_norm[y_cols]= (train_df[y_cols] - means['EUR/MWh']) / stds['EUR/MWh']

dev_df_norm = dev_df.copy()
dev_df_norm[norm_cols] = (dev_df[norm_cols] - means) / stds
y_cols=['y', 'y_day', 'y_week']
dev_df_norm[y_cols]= (dev_df[y_cols] - means['EUR/MWh']) / stds['EUR/MWh']

test_df_norm = test_df.copy()
test_df_norm[norm_cols] = (test_df[norm_cols] - means) / stds
y_cols=['y', 'y_day', 'y_week']
test_df_norm[y_cols]= (test_df[y_cols] - means['EUR/MWh']) / stds['EUR/MWh']

In [38]:
#Save train, dev, test sets
train_df_norm.to_pickle(os.path.join("data", "Datasets_normalized", "train_set_normalized.pkl"))
dev_df_norm.to_pickle(os.path.join("data", "Datasets_normalized", "dev_set_normalized.pkl"))
test_df_norm.to_pickle(os.path.join("data", "Datasets_normalized", "test_set_normalized.pkl"))
#train_df.to_csv(os.path.join("data", "Datasets_2","train_set.csv"), index=False)
#dev_df.to_csv(os.path.join("data", "Datasets_2","dev_set.csv"), index=False)
#test_df.to_csv(os.path.join("data", "Datasets_2","test_set.csv"), index=False)